<a href="https://colab.research.google.com/github/fangjinjie438-pixel/RL-/blob/main/notebooks/run_training_ppo_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training Pipeline
[run_training_ppo_pipeline.ipynb](https://github.com/shibing624/MedicalGPT/blob/main/notebooks/run_training_ppo_pipeline.ipynb)    | [Open In Colab](https://colab.research.google.com/github/shibing624/MedicalGPT/blob/main/notebooks/run_training_ppo_pipeline.ipynb)

# Stage 1: Continue Pretraining

第一阶段：PT(Continue PreTraining)增量预训练，在海量领域文本数据上二次预训练GPT模型，以适配领域数据分布

注意：
1. 此阶段是可选的，如果你没有海量领域文本，可以跳过此阶段，直接进行SFT阶段的有监督微调
2. 我实验发现：做领域知识注入，SFT比PT更高效，也可以跳过PT阶段

| Stage 1: Continue Pretraining   |  [pretraining.py](https://github.com/shibing624/MedicalGPT/blob/main/training/pretraining.py) | [run_pt.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_pt.sh)    |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B
2. 数据集：PT阶段使用的是中文天龙八部小说部分文本和英文书籍部分文本，位于`data/pretrain`文件夹

## 配置运行环境

本地执行可注释以下配置环境的命令，colab执行要打开注释，用于配置环境

colab建议使用T4 GPU训练，设置方式：`代码执行程序 -> 更改运行时类型 -> 运行时类型：Python3，硬件加速器：GPU，GPU类型：T4 -> 保存`

步骤：
1. 下载最新代码到本地
2. 安装依赖包

依赖包如下，保证最新版本：

```
loguru
transformers
sentencepiece
datasets
tensorboard
tqdm
peft
trl
```

In [ ]:
# !pip install --force-reinstall transformers peft accelerate datasets trl

安装完成后，如果依然报错，请点击菜单栏的 `代码执行程序` -> `重新启动会话`，以确保 Python 能够加载最新安装的包。

In [5]:
!git clone --depth 1 https://github.com/shibing624/MedicalGPT.git
%cd MedicalGPT
%ls
!pip install -r requirements.txt

fatal: destination path 'MedicalGPT' already exists and is not an empty directory.
/content/MedicalGPT
CITATION.cff     demo/       notebooks/      requirements.txt  tools/
_config.yml      DISCLAIMER  outputs-pt-v1/  role_play_data/   training/
CONTRIBUTING.md  docs/       README_EN.md    scripts/
data/            LICENSE     README.md       tests/


In [6]:
!pip install --upgrade torchao

## Stage1 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

**以下参数可以根据你的GPU实际情况修改，当前参数是根据Colab的T4单卡GPU（16GB显存）配置的**

In [7]:
%ls ./data/pretrain/

en_article_tail500.txt  fever.txt  tianlongbabu.txt


In [21]:
!rm -rf outputs-sft-v1

In [27]:
!python training/pretraining.py \
    --model_name_or_path Qwen/Qwen2.5-0.5B \
    --train_file_dir ./data/pretrain \
    --validation_file_dir ./data/pretrain \
    --per_device_train_batch_size 3 \
    --per_device_eval_batch_size 3 \
    --do_train \
    --do_eval \
    --use_peft True \
    --seed 42 \
    --bf16 \
    --max_train_samples 20000 \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-4 \
    --warmup_ratio 0.05 \
    --weight_decay 0.01 \
    --logging_strategy steps \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 500 \
    --save_strategy steps \
    --save_total_limit 3 \
    --gradient_accumulation_steps 1 \
    --preprocessing_num_workers 1 \
    --block_size 128 \
    --output_dir outputs-pt-v1 \
    --ddp_timeout 30000 \
    --logging_first_step True \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --ddp_find_unused_parameters False \
    --gradient_checkpointing True

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-05-03 12:33:59.734 | INFO     | __main__:main:364 - Model args: ModelArguments(model_name_or_path='Qwen/Qwen2.5-0.5B', tokenizer_name_or_path=None, load_in_8bit=False, load_in_4bit=False, cache_dir=None, model_revision='main', hf_hub_token=None, use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='auto', trust_remote_code=True)
2026-05-03 12:33:59.734 | INFO     | __main__:main:365 - Data args: DataArguments(dataset_name=None, dataset_config_name=None, train_file_dir='./data/pretrain', validation_file_dir='./data/pretrain', max_train_samples=20000, max_eval_samples=10, streaming=False, block_size=128, overwrite_cache=False, validation_split_percentage=1, preprocessing_num_workers=1, keep_linebreaks=True, packing=True)
2026-05-03 12:33:59.735 | INFO    

In [8]:
%ls -lh outputs-pt-v1

total 28M
-rw-r--r-- 1 root root 1.1K May  3 12:44 adapter_config.json
-rw-r--r-- 1 root root  17M May  3 12:44 adapter_model.safetensors
-rw-r--r-- 1 root root  470 May  3 12:44 all_results.json
-rw-r--r-- 1 root root 2.4K May  3 12:44 chat_template.jinja
drwxr-xr-x 2 root root 4.0K May  3 12:40 checkpoint-500/
drwxr-xr-x 2 root root 4.0K May  3 12:44 checkpoint-845/
-rw-r--r-- 1 root root  262 May  3 12:44 eval_results.json
-rw-r--r-- 1 root root 5.1K May  3 12:44 README.md
drwxr-xr-x 3 root root 4.0K May  3 12:34 runs/
-rw-r--r-- 1 root root  723 May  3 12:44 tokenizer_config.json
-rw-r--r-- 1 root root  11M May  3 12:44 tokenizer.json
-rw-r--r-- 1 root root  21K May  3 12:44 trainer_state.json
-rw-r--r-- 1 root root  228 May  3 12:44 train_results.json


In [9]:
!tree /content/MedicalGPT/outputs-pt-v1

/bin/bash: line 1: tree: command not found


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [11]:
!python tools/merge_peft_adapter.py \
    --base_model Qwen/Qwen2.5-0.5B --lora_model outputs-pt-v1 --output_dir merged-pt/

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Namespace(base_model='Qwen/Qwen2.5-0.5B', tokenizer_path=None, lora_model='outputs-pt-v1', resize_emb=False, output_dir='merged-pt/', hf_hub_model_id='', hf_hub_token=None)
Base model: Qwen/Qwen2.5-0.5B
LoRA model: outputs-pt-v1
Loading LoRA for causal language model (archs=['Qwen2ForCausalLM'])
Loading weights: 100% 290/290 [00:00<00:00, 858.30it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100% 1/1 [00:13<00:00, 13.43s/it]
Done! model saved to merged-pt/


In [12]:
%ls -lh merged-pt/

total 954M
-rw-r--r-- 1 root root 2.4K May  3 13:10 chat_template.jinja
-rw-r--r-- 1 root root 1.3K May  3 13:10 config.json
-rw-r--r-- 1 root root  138 May  3 13:10 generation_config.json
-rw-r--r-- 1 root root 943M May  3 13:10 model.safetensors
-rw-r--r-- 1 root root  697 May  3 13:10 tokenizer_config.json
-rw-r--r-- 1 root root  11M May  3 13:10 tokenizer.json


In [13]:
%cat merged-pt/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage1 增量预训练完成。

# Stage 2: Supervised FineTuning

第二阶段：SFT(Supervised Fine-tuning)有监督微调，构造指令微调数据集，在预训练模型基础上做指令精调，以对齐指令意图，并注入领域知识

| Stage 2: Supervised Fine-tuning | [supervised_finetuning.py](https://github.com/shibing624/MedicalGPT/blob/main/training/supervised_finetuning.py) | [run_sft.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_sft.sh)  |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B 或者 Stage1得到的预训练模型
2. 数据集：SFT阶段使用的是使用的是Belle的1千条抽样数据，位于`data/finetune`文件夹

## Stage2 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

In [18]:
!pwd
!ls

/content/MedicalGPT
CITATION.cff	 demo	     merged-pt	    README.md	      tests
_config.yml	 DISCLAIMER  notebooks	    requirements.txt  tools
CONTRIBUTING.md  docs	     outputs-pt-v1  role_play_data    training
data		 LICENSE     README_EN.md   scripts


In [19]:
%ls ./data/sft

glaive_toolcall_zh_demo.jsonl  sharegpt_zh_1K_format.jsonl
medical_sft_1K_format.jsonl


In [22]:
!python training/supervised_finetuning.py \
    --model_name_or_path merged-pt \
    --train_file_dir ./data/sft \
    --validation_file_dir ./data/sft \
    --per_device_train_batch_size 4 \
    --per_device_eval_batch_size 4 \
    --do_train \
    --do_eval \
    --use_peft True \
    --bf16 \
    --max_train_samples 1000 \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-5 \
    --warmup_ratio 0.05 \
    --weight_decay 0.05 \
    --logging_strategy steps \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 500 \
    --save_strategy steps \
    --save_total_limit 3 \
    --gradient_accumulation_steps 1 \
    --preprocessing_num_workers 1 \
    --output_dir outputs-sft-v1 \
    --ddp_timeout 30000 \
    --logging_first_step True \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --ddp_find_unused_parameters False \
    --gradient_checkpointing True

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-05-03 13:36:23.702 | WARNING  | __main__:__post_init__:191 - You may set max_train_samples = -1 to run all samples in production.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-05-03 13:36:23.910 | INFO     | __main__:main:352 - Model args: ModelArguments(model_name_or_path='merged-pt', load_in_8bit=False, load_in_4bit=False, tokenizer_name_or_path=None, cache_dir=None, model_revision='main', hf_hub_token=None, use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='auto', trust_remote_code=True, rope_scaling=None, flash_attn=False, shift_attn=False, neft_alpha=0)
2026-05-03 13:36:23.910 | INFO     | __main__:main:353 - Data args: DataArguments(dataset_name=None, dataset_config_name=None, train_file_dir='./data/sft', validation_file_dir='./data/sft', max_train_samples=1000, max_eval_samples=10, i

In [23]:
%ls -lh outputs-sft-v1

total 28M
-rw-r--r-- 1 root root 1.1K May  3 13:48 adapter_config.json
-rw-r--r-- 1 root root  17M May  3 13:48 adapter_model.safetensors
-rw-r--r-- 1 root root  428 May  3 13:48 all_results.json
-rw-r--r-- 1 root root 2.4K May  3 13:48 chat_template.jinja
drwxr-xr-x 2 root root 4.0K May  3 13:48 checkpoint-250/
-rw-r--r-- 1 root root  220 May  3 13:48 eval_results.json
-rw-r--r-- 1 root root 5.1K May  3 13:48 README.md
drwxr-xr-x 3 root root 4.0K May  3 13:36 runs/
-rw-r--r-- 1 root root  733 May  3 13:48 tokenizer_config.json
-rw-r--r-- 1 root root  11M May  3 13:48 tokenizer.json
-rw-r--r-- 1 root root 6.3K May  3 13:48 trainer_state.json
-rw-r--r-- 1 root root  228 May  3 13:48 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [24]:
!python tools/merge_peft_adapter.py \
    --base_model merged-pt --lora_model outputs-sft-v1 --output_dir merged-sft/

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Namespace(base_model='merged-pt', tokenizer_path=None, lora_model='outputs-sft-v1', resize_emb=False, output_dir='merged-sft/', hf_hub_model_id='', hf_hub_token=None)
Base model: merged-pt
LoRA model: outputs-sft-v1
Loading LoRA for causal language model (archs=['Qwen2ForCausalLM'])
Loading weights: 100% 290/290 [00:00<00:00, 776.73it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100% 1/1 [00:05<00:00,  5.66s/it]
Copied tokenizer/config files from merged-pt: ['tokenizer_config.json', 'tokenizer.json', 'chat_template.jinja', 'generation_config.json']
Done! model saved to merged-sft/


In [25]:
%ls -lh merged-sft/

total 954M
-rw-r--r-- 1 root root 2.4K May  3 13:10 chat_template.jinja
-rw-r--r-- 1 root root 1.3K May  3 13:48 config.json
-rw-r--r-- 1 root root  138 May  3 13:10 generation_config.json
-rw-r--r-- 1 root root 943M May  3 13:49 model.safetensors
-rw-r--r-- 1 root root  697 May  3 13:10 tokenizer_config.json
-rw-r--r-- 1 root root  11M May  3 13:10 tokenizer.json


In [26]:
%cat merged-sft/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage2 SFT训练完成。

# Stage 3: Reward Modeling

第三阶段：RM(Reward Model)奖励模型建模，构造人类偏好排序数据集，训练奖励模型，用来对齐人类偏好，主要是"HHH"原则，具体是"helpful, honest, harmless"

| Stage 3: Reward Modeling        |  [reward_modeling.py](https://github.com/shibing624/MedicalGPT/blob/main/training/reward_modeling.py) | [run_rm.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_rm.sh)    |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B 或者 Stage2得到的SFT模型
2. 数据集：RM阶段使用的是医疗reward数据，抽样了500条，位于`data/reward`文件夹

## Stage3 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

In [27]:
%ls ./data/reward/

dpo_zh_500.jsonl  toolcall_dpo_zh_demo.jsonl


In [30]:
!python training/reward_modeling.py \
    --model_name_or_path merged-sft \
    --train_file_dir ./data/reward \
    --validation_file_dir ./data/reward \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 1 \
    --do_train \
    --use_peft True \
    --seed 42 \
    --max_train_samples 1000 \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-5 \
    --warmup_ratio 0.05 \
    --weight_decay 0.001 \
    --logging_strategy steps \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 500 \
    --save_strategy steps \
    --save_total_limit 3 \
    --max_source_length 256 \
    --max_target_length 256 \
    --output_dir outputs-rm-v1 \
    --overwrite_output_dir \
    --ddp_timeout 30000 \
    --logging_first_step True \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype float16 \
    --fp16 \
    --device_map auto \
    --report_to tensorboard \
    --ddp_find_unused_parameters False \
    --remove_unused_columns False \
    --gradient_checkpointing True

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-05-03 14:04:10.646 | INFO     | __main__:main:342 - Model args: ModelArguments(model_name_or_path='merged-sft', tokenizer_name_or_path=None, load_in_4bit=False, load_in_8bit=False, cache_dir=None, use_fast_tokenizer=False, torch_dtype='float16', device_map='auto', trust_remote_code=True)
2026-05-03 14:04:10.647 | INFO     | __main__:main:343 - Data args: DataArguments(dataset_name=None, dataset_config_name=None, train_file_dir='./data/reward', validation_file_dir='./data/reward', max_source_length=256, max_target_length=256, max_train_samples=1000, max_eval_samples=10, overwrite_cache=False, validation_split_percentage=1, preprocessing_num_workers=4)
2026-05-03 14:04:10.647 | INFO     | __main__:main:344 - Training args: TrainingArguments(
accelerator_config={'

In [31]:
%ls -lh outputs-rm-v1

total 28M
-rw-r--r-- 1 root root 1.1K May  3 14:10 adapter_config.json
-rw-r--r-- 1 root root  17M May  3 14:10 adapter_model.safetensors
-rw-r--r-- 1 root root  483 May  3 14:10 all_results.json
-rw-r--r-- 1 root root 2.4K May  3 14:10 chat_template.jinja
drwxr-xr-x 2 root root 4.0K May  3 14:10 checkpoint-364/
-rw-r--r-- 1 root root  290 May  3 14:10 eval_results.json
-rw-r--r-- 1 root root 5.1K May  3 14:10 README.md
drwxr-xr-x 3 root root 4.0K May  3 14:05 runs/
-rw-r--r-- 1 root root  707 May  3 14:10 tokenizer_config.json
-rw-r--r-- 1 root root  11M May  3 14:10 tokenizer.json
-rw-r--r-- 1 root root 9.9K May  3 14:10 trainer_state.json
-rw-r--r-- 1 root root  213 May  3 14:10 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [33]:
!python tools/merge_peft_adapter.py \
    --base_model merged-sft --lora_model outputs-rm-v1 --output_dir merged-rm/

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Namespace(base_model='merged-sft', tokenizer_path=None, lora_model='outputs-rm-v1', resize_emb=False, output_dir='merged-rm/', hf_hub_model_id='', hf_hub_token=None)
Base model: merged-sft
LoRA model: outputs-rm-v1
Loading LoRA for sequence classification model
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:05<00:00, 50.49it/s]
[transformers] Qwen2ForSequenceClassification LOAD REPORT from: merged-sft
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100% 1/1 [00:13<00:00, 13.62s/it]
Copied tokenizer/config files from merged-sft: ['tokenizer_config.json', 'tokenizer

In [34]:
%ls -lh merged-rm/

total 1.9G
-rw-r--r-- 1 root root 2.4K May  3 13:10 chat_template.jinja
-rw-r--r-- 1 root root 1.4K May  3 14:19 config.json
-rw-r--r-- 1 root root  138 May  3 13:10 generation_config.json
-rw-r--r-- 1 root root 1.9G May  3 14:19 model.safetensors
-rw-r--r-- 1 root root  697 May  3 13:10 tokenizer_config.json
-rw-r--r-- 1 root root  11M May  3 13:10 tokenizer.json


In [35]:
%cat merged-rm/config.json

{
  "architectures": [
    "Qwen2ForSequenceClassification"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float32",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "id2label": {
    "0": "LABEL_0"
  },
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "label2id": {
    "LABEL_0": 0
  },
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_atten

Stage3 奖励建模第一次训练完成。

# Stage 4: Reinforcement Learning Training

第四阶段：RL(Reinforcement Learning)基于人类反馈的强化学习(RLHF)，用奖励模型来训练SFT模型，生成模型使用奖励或惩罚来更新其策略，以便生成更高质量、更符合人类偏好的文本

| Stage 4: Reinforcement Learning |  [ppo_training.py](https://github.com/shibing624/MedicalGPT/blob/main/training/ppo_training.py) | [run_ppo.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_ppo.sh)    |


#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型、奖励模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B 或者 Stage2得到的SFT模型
2. 奖励模型：使用的是`OpenAssistant/reward-model-deberta-v3-large-v2` 或者 Stage3得到的BERT类或者GPT类奖励模型
3. 数据集：RL阶段的数据可以复用SFT的数据集，使用的是Belle的1千条抽样数据，位于`data/finetune`文件夹

## Stage4 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载生成模型和tokenizer，加载奖励模型和其tokenizer
5. 开始训练并评估
6. 查看训练结果

以下参数可以根据你的GPU实际情况修改，当前参数是根据Colab的T4单卡GPU（16GB显存）配置的。

In [41]:
! CUDA_VISIBLE_DEVICES=0 python training/ppo_training.py \
    --sft_model_path ./merged-sft \
    --reward_model_path ./merged-rm \
    --template_name qwen \
    --torch_dtype bfloat16 \
    --bf16 True \
    --train_file_dir ./data/sft \
    --validation_file_dir ./data/sft \
    --max_source_length 256 \
    --response_length 1000 \
    --do_train \
    --save_steps 50 \
    --output_dir outputs-ppo-v1 \
    --num_train_epochs 3 \
    --report_to tensorboard

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-05-03 14:45:15.879 | INFO     | __main__:main:70 - Parse args: RLOOArguments(sft_model_path='./merged-sft', reward_model_path='./merged-rm', dataset_name=None, dataset_config=None, dataset_train_split='train', dataset_test_split='test', train_file_dir='./data/sft', validation_file_dir='./data/sft', template_name='qwen', max_source_length=256)
2026-05-03 14:45:15.879 | INFO     | __main__:main:71 - Training args: RLOOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.05,
bf16=True,
bf16_full_eval=False,
cache_implementation=None,
chat_templat

In [42]:
!nvidia-smi

Sun May  3 14:49:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [37]:
%ls -lh outputs-ppo-v1

ls: cannot access 'outputs-ppo-v1': No such file or directory


模型训练结果：
- use_peft=False,默认是使用全参训练，模型保存的就是`model-00001-of-00002.safetensors`等文件，配置文件是`config.json`
- use_peft=True, 则使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/trl`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/trl --host 0.0.0.0 --port 8009`

In [23]:
%ls -lh outputs-ppo-v1/

ls: cannot access 'outputs-ppo-v1/': No such file or directory


In [24]:
%cat outputs-ppo-v1/config.json

cat: outputs-ppo-v1/config.json: No such file or directory


Stage4 RL第一次训练完成。

**至此一个完整的4阶段训练流程演示完成。**

实际操作中Stage3和Stage4可以反复多次，直到RL得到的最后模型满足评估要求。

RLHF过程可以把SFT模型当成一个初始化模型，RM模型当做指导老师，使用RL(PPO)调教SFT模型生成指导老师最满意的结果，如果小学老师满意了，我们就再训练一个中学老师，继续指导，中学老师满意了，就训练一个大学老师，这样不断迭代，使得生成模型的质量达到甚至超过人工撰写的天花板。

RLHF训练不易，此项目提供给大家一种实现的方法和参考，希望抛砖引玉，共同促进中文开源LLM发展。

# Test

In [25]:
!python demo/inference.py --base_model merged-ppo-v1
# 或在shell中运行
# !python demo/inference.py --base_model merged-ppo-v1 --interactive

Namespace(base_model='merged-ppo-v1', lora_model='', tokenizer_path=None, system_prompt='', stop_str='', repetition_penalty=1.0, max_new_tokens=512, data_file=None, interactive=False, single_tune=False, temperature=0.7, output_file='./predictions_result.jsonl', eval_batch_size=4, resize_emb=False, load_in_8bit=False, load_in_4bit=False)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '401 Unauthorized' for url 'https://huggingface.co/merged-ppo-v1/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

The above exception was the direct cause of the following exception:

Traceback (most recent call l

Input:介绍下南京
Response:  南京市位于江苏省西南部，是全国首批历史文化名城、国家中心城市和自由贸易试验区。

完。
